In [ ]:
import os
import sys
import subprocess
import glob

# 🎛️ SET THIS TO TRUE FOR TPU, FALSE FOR GPU
FORCE_TPU = True

def repair_environment():

    if FORCE_TPU:
        print("🔍 Starting High-Speed TPU Repair...")
    
        # 1. Faster Uninstallation 
        print("🧹 Wiping libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", 
                        "torch", "torch_xla", "torchvision", "numpy", "tensorflow"], 
                       capture_output=True)
    
        # 2. Parallel/Bulk Installation
        print("📥 Installing Synced TPU Stack...")
        common_args = ["install", "-q", "--no-warn-script-location"]
        
        if FORCE_TPU or glob.glob("/dev/accel*"):
            cmd = [
                sys.executable, "-m", "pip", *common_args,
                "torch==2.8.0", 
                "torchvision==0.23.0", 
                "torch_xla[tpu]==2.8.0",
                # 🩹 THE FIX: Unpinned numpy, pyarrow, and fsspec to allow Numpy 2.0+ compatibility
                "numpy", "pyarrow", "fsspec", 
                "datasets", "transformers", "huggingface_hub", "wandb", 
                "cloud-tpu-client", "scikit-learn", "pandas<3.0.0", 
                "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
                "--extra-index-url", "https://download.pytorch.org/whl/cpu"
            ]
            subprocess.check_call(cmd)
        else:
            # Fallback
            cmd = [
                sys.executable, "-m", "pip", *common_args, "-U",
                "torch", "datasets", "pyarrow", "transformers", "huggingface_hub", "fsspec", "wandb", "scipy", "numpy", "pandas<3.0.0"
            ]
            subprocess.check_call(cmd)
    
        print("\n✅ TPU REPAIR COMPLETE.")
        print("⚠️ Click 'Run' -> 'Restart Session' NOW.")

    else:
        print("🔍 Starting Robust GPU Repair...")
    
        # 1. Clean Wipe
        print("🧹 Wiping conflicting libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", 
                        "torch", "torchvision", "torchaudio"], 
                       capture_output=True)
    
        # 2. Setup Arguments
        common_args = ["install", "-q", "--no-warn-script-location"]
        
        try:
            print("📥 Installing GPU/CUDA Stack...")
            
            print("   ⚡ Part 1: PyTorch Core...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args,
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"
            ])
            
            print("   ⚡ Part 2: Transformers & Data...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args, "-U",
                "datasets", "transformers", "huggingface_hub", 
                "wandb", "pandas<3.0.0" 
            ])
    
            print("\n✅ GPU REPAIR COMPLETE.")
            print("⚠️ MANDATORY: Click 'Run' -> 'Restart Session' NOW.")
    
        except subprocess.CalledProcessError as e:
            print(f"\n❌ Installation failed. Error: {e}")
            print("💡 Try manually restarting the session and running this cell again.")

if __name__ == "__main__":
    repair_environment()

In [ ]:
!python universal_llm_trainer.py

In [ ]:
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from dataclasses import dataclass
from transformers import AutoTokenizer
import json
import multiprocessing

class ConfigJson():
    def __init__(self, **kwargs):
        # Assign attributes from keyword arguments
        for key, value in kwargs.items():
            setattr(self, key, value)

    @classmethod
    def from_json(cls, json_path):
        with open(json_path, "r") as file:
            data = json.load(file)
        return cls(**data)



class SpanMLMCollator:

    # Define Init
    # Most of the things are just stored in the config lwk
    # Imma just pull from here
    def __init__(self, config = None, tokenizer = None, mlm_probability = None, mlm_use_span_masking = None, mlm_span_length = None):
        
        # Defaults
        self.mlm_probability = 0.15
        self.mlm_use_span_masking = False
        self.mlm_span_length = 3
        self.tokenizer = None

        # Override with Config (if provided)
        if (config is not None):
            if (isinstance(config, str)):
                try:
                    config = ConfigJson.from_json(config)
                except Exception as e:
                    raise ValueError(f"Blud was not a json. Either an AutoConfig (HF) or .json. Error: {e}")

            self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)
            self.mlm_probability = config.mlm_probability
            self.mlm_use_span_masking = config.mlm_use_span_masking
            self.mlm_span_length = config.mlm_span_length

        if tokenizer is not None:
            self.tokenizer = tokenizer
        if mlm_probability is not None:
            self.mlm_probability = mlm_probability
        if mlm_use_span_masking is not None:
            self.mlm_use_span_masking = mlm_use_span_masking
        if mlm_span_length is not None:
            self.mlm_span_length = mlm_span_length
            
        if self.tokenizer is None:
            raise ValueError("Tokenizer is None. Either pass in the config or tokenizer")

        # AI HELP:
        # Create vocab for legal words to replace
        valid_ids = [i for i in range(len(self.tokenizer)) if i not in self.tokenizer.all_special_ids]
        self.valid_vocab = torch.tensor(valid_ids)

    # __call__() function returning a DataLoader with Span_masking
    def __call__(self, data):

        # Convert List of Dictionaries into 1 large tensor
        # inside, convert extract input_ids from dictionary
        batch_input_ids = [d["input_ids"] for d in data]
        input_ids = torch.tensor(batch_input_ids)
        labels = input_ids.clone()

        # Set mlm_prob
        mlm_prob = self.mlm_probability
        if (self.mlm_use_span_masking):
            mlm_prob /= self.mlm_span_length

        # Create the Masking Tensor
        masked_tensor = torch.full(input_ids.shape, mlm_prob, dtype=torch.float32)


        # HUH???


        # =====================================================================
        # THE GUARD STEP: Protect special tokens from being masked
        # =====================================================================
        # 1. Ask the tokenizer which tokens in each row are special (CLS, SEP, PAD)
        special_tokens_mask = [
            self.tokenizer.get_special_tokens_mask(seq, already_has_special_tokens=True) 
            for seq in input_ids.tolist()
        ]
        
        # 2. Convert that nested list into a PyTorch Boolean tensor
        special_tokens_map = torch.tensor(special_tokens_mask, dtype=torch.bool)
        
        # 3. Overwrite the probability in masked_tensor to 0.0 wherever special_tokens_map is True
        masked_tensor.masked_fill_(special_tokens_map, value=0.0)
        # =====================================================================


        # Apply Bernoulli to get 1s and 0s for the masks
        masked_tensor = torch.bernoulli(masked_tensor).bool()

        temp_clone = masked_tensor.clone()
        # Apply rolling for span masking
        if (self.mlm_use_span_masking):
            for i in range (1, self.mlm_span_length):
                rolled_tensor = torch.roll(temp_clone, shifts = i)
                rolled_tensor[:,:i] = False
                masked_tensor = masked_tensor | rolled_tensor

        # Mask all untampered tokens with -100
        labels[~masked_tensor] = -100
        
        # Create new Tensor w/ random values from 0-1
        type_tensor = torch.rand(input_ids.shape)

        # Create 80% mask_token_tensor
        mask_token_tensor = (type_tensor <= .8) & masked_tensor

        # Create 10% corrupted_token_tensor
        corrupted_token_tensor = (type_tensor > .8) & (type_tensor <= .9) & masked_tensor

        # Apply Mask tokens to input_ids
        input_ids[mask_token_tensor] = self.tokenizer.mask_token_id

        # LOOK HERE ##################################################
        
        # Apply corrupted tokens to input_ids
        num_to_replace = corrupted_token_tensor.sum().item()

        # Generate random indices for the valid_vocab_tensor
        indices = torch.randint(0,len(self.valid_vocab), (num_to_replace,))

        # Take the words form valid_vocab
        random_words = self.valid_vocab[indices]

        # Apply the words to the input_ids
        input_ids[corrupted_token_tensor] = random_words

        # ############################################################

        # Return Dictionary
        return {"input_ids": input_ids, "labels": labels}

In [ ]:
%%writefile parallel_hardware_trainer.py
import os
import sys
import importlib
import time
import json
import site
import torch
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass, field # Added field here
import multiprocessing
import SpanMLMCollator
from typing import Optional, List, Union, ClassVar
import warnings

# Disable Progress Bars
try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()
except ImportError:
    pass

# Ensure PJRT runtime gets selected, not XRT
for key in ["XRT_TPU_CONFIG", "PJRT_SELECT_DEVICE", "TPU_PROCESS_ADDRESSES"]:
    os.environ.pop(key, None)
os.environ["PJRT_DEVICE"] = "TPU"
# Add the framework quarantine just in case!
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" 

# Prevent C++ thread deadlocks during 10B token streaming
os.environ["OMP_NUM_THREADS"] = "1"
import pyarrow as pa
pa.set_cpu_count(1)
pa.set_io_thread_count(1)

# Tell everything to stay away from the TPU except PyTorch
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"

# Force Path Refresh
if 'site' in sys.modules:
    importlib.reload(site)


@dataclass
class HardwareConfig:
    HARDWARE_PROFILES = {
        # Key: {micro_batch, world_size, target_gbs}
        "v5e-8":  {"mb": 1, "ws": 8, "target": 64}, # TPU Pod
        "v5e-1":  {"mb": 16, "ws": 1, "target": 128}, # Single TPU
        "v6e-1":  {"mb": 32, "ws": 1, "target": 128}, # Next-gen TPU
        "t4*2":   {"mb": 8,  "ws": 2, "target": 128}, # Dual T4 (Kaggle)
        "t4":     {"mb": 8,  "ws": 1, "target": 128}, # Single T4
        "l4":     {"mb": 24, "ws": 1, "target": 128}, # Single L4
        "p100":   {"mb": 16, "ws": 1, "target": 128}, # Single P100
        "a100":   {"mb": 64, "ws": 1, "target": 128}, # Single A100
        "cpu":    {"mb": 2,  "ws": 1, "target": 128}, # Local CPU
    }
    hardware_string: str = "v5e-8"
    hf_token: str = "" 
    base_repo_id = "JamesResearch1216/HELM-Architecture"
    repo_ver_override: Optional[int] = None

    # These will be overwritten, but just place_holders for now
    world_size: int = 1
    batch_size: int = 16
    grad_accum_steps: int = 1
    device_type: str = "cpu"



@dataclass
# This MLMDataConfig Class 
class MLMDataConfig:
    repo_id: str = "JamesResearch1216/HELM-Processed-Data-10B"
    curriculum: bool = True
    curriculum_subset_names: List[str] = field(
        default_factory=lambda: ["seq_1024", "seq_2048", "seq_4096"]
    )
    train_split: str = "train"
    validation_split: str = "validation" 
    tokenizer_name: str = "answerdotai/ModernBERT-base"
    max_seq_len: int = 1024
    batch_size: int = 32
    mlm_probability: float = 0.3
    mlm_use_span_masking: bool = True
    mlm_span_length: int = 3

class MLMDataStrategy:

    # Initialize (Different for each hardware)
    def __init__(self, rank = 0, world_size = 1, is_tpu = False, config: Optional[MLMDataConfig] = None, hf_token=None):
        self.rank = rank
        self.world_size = world_size
        self.is_tpu = is_tpu
        self.config = config
        self.hf_token = hf_token
    
    # Create get_mlm_data_loader function
    # Note: This only bascially works with the dataset created by prepare_data.py

    def get_mlm_data_loader(self, collate_fn = None, curriculum_level = 0, is_train = True):

        # Set up Dataset Differently for curriculum
        if (self.config.curriculum):
            dataset = load_dataset(
                path = self.config.repo_id, 
                name = self.config.curriculum_subset_names[curriculum_level], 
                split = self.config.train_split if is_train else self.config.validation_split,
                streaming = True,
                token=self.hf_token
            )
        else:
            # Else just load normally
            dataset = load_dataset(
                path = self.config.repo_id, 
                name = self.config.dataset_subset, 
                split = self.config.train_split if is_train else self.config.validation_split,
                streaming = True,
                token=self.hf_token
            )

        # IF we're Parallel Processing, Shard the Data so that each device gets a different slice of data
        if self.world_size > 1:
            dataset = dataset.shard(num_shards = self.world_size, index = self.rank)

        # Shuffle after you shard
        dataset = dataset.shuffle(buffer_size = 10000)

        # set workers = 0 for TPUs else actually the real number of CPU cores
        # this was in the 1.x_model_trainer series of scripts
        workers = 0 if self.is_tpu else max(1, multiprocessing.cpu_count()-1)

        data_loader = DataLoader(
            dataset, 
            batch_size = self.config.batch_size, 
            num_workers = workers, 
            drop_last = True, 
            pin_memory = True if torch.cuda.is_available() else False,  
            collate_fn = collate_fn
        )

        # Return data_loader
        return data_loader

class HardwareDriver:

    # Initialize Hardware
    def __init__(self, hw_config: HardwareConfig, data_config: MLMDataConfig):
        self.hw_config = hw_config
        self.data_config = data_config
        # Call _parse_hardware here
        self._parse_hardware()
    

    def _parse_hardware(self):
        # get hardware_string (with formatting)
        hardware_string = self.hw_config.hardware_string.lower().replace(" ", "")

        # Ensure that the hardware_string is valid
        profile = None
        for key in self.hw_config.HARDWARE_PROFILES:
            if (key in hardware_string):
                profile = self.hw_config.HARDWARE_PROFILES[key]
                break

        # Default to cpu if none were matching
        if not profile:
            profile = self.hw_config.HARDWARE_PROFILES["cpu"]
            warnings.warn("hardware_string did not match any in HARDWARE_PROFILES. Using \"cpu\"", UserWarning)

        # Add attributes to config
        # get num_workers
        self.hw_config.world_size = profile["ws"]
        # define # of microbatches to equate to batch_size = 128
        # (used during gradient accumulation)
        self.hw_config.batch_size = profile["mb"]
        self.data_config.batch_size = profile["mb"]

        # Calculate gradient accumulation steps to get to target batch_size (128)
        target = profile["target"]
        self.hw_config.grad_accum_steps = max(1, target // (profile["mb"] * profile["ws"]))

        # Get device type (save to hw_config)
        if "tpu" in hardware_string:
            self.hw_config.device_type = "tpu"
        elif any(x in hardware_string for x in ["gpu", "cuda", "a100", "p100", "h100" "t4", "l4"]):
            self.hw_config.device_type = "cuda"
        else:
            self.hw_config.device_type = "cpu"
    
    # Launch function: Spawn all the workers and make them run the worker_function
    def launch(self, worker_fn):

        # Define these for convience
        world_size = self.hw_config.world_size
        device = self.hw_config.device_type
        
        # If parallel processing
        if world_size > 1:
            if device == "tpu":
                import torch_xla.distributed.xla_multiprocessing as xmp
                xmp.spawn(worker_fn, args=(self.hw_config, self.data_config), start_method='spawn')
            elif device == "cuda":
                import random
                import torch.multiprocessing as mp
                
                # Set up Multi-GPU network
                os.environ['MASTER_ADDR'] = 'localhost'
                os.environ['MASTER_PORT'] = str(random.randint(10001, 19999))


                mp.spawn(worker_fn, args=(self.hw_config, self.data_config), nprocs=ws)
        else:
            # Single Device Execution (Rank 0)
            worker_fn(0, self.hw_config, self.data_config)



def train_worker(rank, hw_config, data_config):

    import sys
    import traceback
    
    try:
        import datasets
        datasets.config.TF_AVAILABLE = False
        
        import torch
        import torch.nn as nn
        import torch.optim as optim
        from transformers import AutoTokenizer
        import SpanMLMCollator
        from model import HELMConfig, HELMForMaskedLM
        

        # Default for Data Collator
        is_tpu = False

        # Load correct packages and get device
        if hw_config.device_type == "tpu":
            # Lazy Load even more for TPU
            import torch_xla.core.xla_model as xm
            import torch_xla.distributed.parallel_loader as pl
            import torch_xla.runtime as xr

            device = xm.xla_device()
            is_tpu = True

            # get real world size just in case
            real_world_size = xr.world_size()
            if hw_config.world_size != real_world_size:
                if rank == 0:
                    print(f"⚠️ CONFIG MISMATCH: Adjusting world size to {real_world_size}")
                hw_config.world_size = real_world_size
        
        elif hw_config.device_type == "cuda":
            # Set cuda device to torch
            torch.cuda.set_device(rank)
            device = torch.device(f"cuda:{rank}")
            # Initialize Distributed comm framework
            # acts like a rendezvous
            # NVIDIA Collective Communications Library (nccl)
            if hw_config.world_size > 1:
                import torch.distributed as dist
                dist.init_process_group("nccl", rank=rank, world_size=hw_config.world_size)
       
        else:
            device = torch.device("cpu")
        

        if rank == 0:
            print(f"Rank 0 is online: {device}.")
        
        # Define Collator
        tokenizer = AutoTokenizer.from_pretrained(
            data_config.tokenizer_name, token=hw_config.hf_token
        )
        
        collator = SpanMLMCollator.SpanMLMCollator(
            config = data_config, tokenizer = tokenizer
        )

        # Define data_strat
        data_strat = MLMDataStrategy(
            rank = rank, world_size = hw_config.world_size, is_tpu = is_tpu,config = data_config, hf_token=hw_config.hf_token
        ) 

        # Define dataloader
        train_loader = data_strat.get_mlm_data_loader(
            collate_fn = collator, curriculum_level = 0, is_train = True
        )

        # if TPU is being used, apply the ParallelLoader().per_device_loader()
        if is_tpu:
            loader = pl.ParallelLoader(train_loader, [device]).per_device_loader(device)
        else:
            loader = train_loader
        
        # Initialize and Force Weight Initialization
        helm_config = HELMConfig(
            vocab_size=len(tokenizer),
            pad_token_id=tokenizer.pad_token_id,
            max_position_embeddings=data_config.max_seq_len,
        )
        model = HELMForMaskedLM(helm_config).to(device)

        # Initialize Weights
        model.apply(model._init_weights)

        # Require DDP to Wrap the model
        if hw_config.device_type == "cuda" and hw_config.world_size > 1:
            from torch.nn.parallel import DistributedDataParallel as DDP
            model = DDP(model, device_ids=[rank])
    
        # Define Optimizer #####
        optimizer = optim.AdamW(model.parameters(), lr = 1e-4)
        # put model into training mode
        loss_fct = nn.CrossEntropyLoss()
        model.train()

        # Zero the gradient
        optimizer.zero_grad()
        
        # Loop through each batch
        for step, batch in enumerate(loader):

            # TEMP: Break after 10 steps
            if step >= 10:
                break
            
            # Get Batch's Input Ids and attach it to device
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            attention_mask = (input_ids != tokenizer.pad_token_id).long().to(device)
            
            # Catch custom architecture outputs
            logits, aux_loss = model(input_ids=input_ids, attention_mask=attention_mask, current_step=step)
            
            ce_loss = loss_fct(logits.view(-1, helm_config.vocab_size), labels.view(-1))
            total_loss = (ce_loss + aux_loss) / hw_config.grad_accum_steps
            total_loss.backward()

            # Once Gradient has been accumulated, step the model and the optimizer
            if (step + 1) % hw_config.grad_accum_steps == 0:
                if is_tpu:
                    xm.optimizer_step(optimizer)
                    xm.mark_step()
                else:
                    optimizer.step()

                # Zero the gradient
                optimizer.zero_grad()
                
                # Use 1 device (rank = 0) to calculate the real loss
                if rank == 0:
                    print(f"Step {step+1} | Loss: {total_loss.item()*hw_config.grad_accum_steps:.4f} (Aux: {aux_loss:.4f})")
        
        # Destroy 
        if hw_config.device_type == "cuda" and hw_config.world_size > 1:
            dist.destroy_process_group()
    
    except Exception as e:
        print(f"\n❌ FATAL WORKER ERROR ON RANK {rank}:")
        traceback.print_exc()
        return


if __name__ == "__main__":
    # Ensure environment is primed for TPU PJRT
    for key in ["XRT_TPU_CONFIG", "PJRT_SELECT_DEVICE", "TPU_PROCESS_ADDRESSES"]:
        os.environ.pop(key, None)
    os.environ["PJRT_DEVICE"] = "TPU"

    import dataclasses

    HW_CFG = HardwareConfig(hardware_string="v5e-8 TPU")
    DATA_CFG = MLMDataConfig()
    
    driver = HardwareDriver(HW_CFG, DATA_CFG)
    driver.launch(train_worker)

In [ ]:
%%writefile model.py

##################################################
# Defines the HELM V1 architecture
# Inherited the PretrainedConfig and PreTrainedModel
# Utilizes many of the concepts found in Nvidia's 2024 nGPT architecture
# Initializes the Weights
# Uses 36 to approx. sqrt(1280)
# Uses 0.028 to approx. 1/sqrt(280)
##################################################

import os
import json
import torch
import numpy as np
from safetensors.torch import load_file
import math
from math import sqrt
import random
import torch.nn.functional as F
import torch.nn as nn
from transformers import AutoTokenizer
from transformers import PretrainedConfig, PreTrainedModel



# modified justnorm() function
# better than F.normalize(), max() causes micro walls during gradient descent
# better than nGPT's version, prevents division by 0 error
def justnorm(x, dim = -1, eps = 1e-12):
    res = x / (x.norm(p=2, dim=dim, keepdim=True) + eps)
    return res

# Hugging Face Core Method (for future deployment)
class HELMConfig(PretrainedConfig):

    model_type = "helm"

    def __init__(
        self,
        hidden_size = 1280,
        sqrt_hidden_size = 36,
        max_position_embeddings = 4096,
        initializer_range = 0.028,
        num_hidden_layers = 12,
        num_attention_heads = 20,
        rope_theta = 160000,
        intermediate_size = 3456,
        norm_eps = 1e-12,
        hidden_act = "swiglu",
        swiglu_s_init = 1.0,
        tokenizer_path = "answerdotai/ModernBERT-base",
        vocab_size = 50368,
        bos_token_id = 50281,
        eos_token_id = 50282,
        pad_token_id = 50283,
        mask_token_id = 50284,
        unk_token_id = 50285,
        mlm_probability = 0.3,
        mlm_use_span_masking = True,
        mlm_span_length = 3,
        num_router_latents = 4,
        max_active_heads = 6,
        num_permanent_heads = 2,
        selection_threshold_init = 0.5,
        router_init_scale = 10.0,
        jitter_noise = 0.01,
        sparsity_lambda = 0.01, 
        router_aux_loss_coeff = 0.02,
        router_grad_clip = 0.05,    
        sparsity_warm_up_steps = 2000,
        use_sigmoid_scaling = False,
        ngpt_sqk_init_value = 1.0,  
        ngpt_sqk_init_scale = 0.028,
        ngpt_alpha_value_attn = 0.05,
        ngpt_alpha_scale_attn = 0.028,
        ngpt_alpha_value_mlp = 0.05,
        ngpt_alpha_scale_mlp = 0.028,
        ngpt_suv_value = 1.0,
        ngpt_suv_scale = 1.0,
        ngpt_sz_init_value = 1.00,
        ngpt_sz_init_scale = 0.028,
        bias = False,
        **kwargs 
    ):  
        self.hidden_size = hidden_size
        self.sqrt_hidden_size = sqrt_hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers 
        self.num_attention_heads = num_attention_heads
        self.rope_theta = rope_theta
        self.intermediate_size = intermediate_size
        self.norm_eps = norm_eps
        self.hidden_act = hidden_act 
        self.swiglu_s_init = swiglu_s_init
        self.tokenizer_path = tokenizer_path
        self.vocab_size = vocab_size
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.unk_token_id = unk_token_id
        self.mlm_probability = mlm_probability
        self.mlm_use_span_masking = mlm_use_span_masking
        self.mlm_span_length = mlm_span_length
        self.num_router_latents = num_router_latents
        self.max_active_heads = max_active_heads
        self.num_permanent_heads = num_permanent_heads
        self.num_elastic_heads = max_active_heads - num_permanent_heads
        self.selection_threshold_init = selection_threshold_init
        self.router_init_scale = router_init_scale
        self.jitter_noise = jitter_noise
        self.sparsity_lambda = sparsity_lambda
        self.router_aux_loss_coeff = router_aux_loss_coeff
        self.router_grad_clip = router_grad_clip     
        self.sparsity_warm_up_steps = sparsity_warm_up_steps
        self.use_sigmoid_scaling = use_sigmoid_scaling
        self.ngpt_sqk_init_value = ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = ngpt_sqk_init_scale
        self.ngpt_alpha_value_attn = ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = ngpt_alpha_scale_mlp
        self.ngpt_suv_value = ngpt_suv_value
        self.ngpt_suv_scale = ngpt_suv_scale
        self.ngpt_sz_init_value = ngpt_sz_init_value
        self.ngpt_sz_init_scale = ngpt_sz_init_scale
        self.bias = bias

        super().__init__(**kwargs)


# Define Embedding Layer
class HELMEmbedding(nn.Module):

    # Initialize Embedding Layer
    def __init__(self, config):
        super().__init__()

        # Embedding Matrix size() : [vocab_size, hidden_size]
        self.word_embeddings = nn.Embedding(
            config.vocab_size, 
            config.hidden_size, 
            padding_idx=config.pad_token_id
        )
        
    # Forward Pass (yes, its literally 3 lines)
    def forward(self, input_ids):
        
        # Map input_ids from Word Embeddings
        word_embeds = self.word_embeddings(input_ids)

        # Normalize (an nGPT must to allow cos. sim. to work)
        embeddings = justnorm(word_embeds)

        # Return
        return embeddings



# NOVEL: Multi-Latent Summary Router to decide which heads to use
class HELMMultiViewRouter(nn.Module):
   
    # Initialize the following:
    #   - Summary Query Matrix (q_down_proj)
    #   - Latent Importance Weights (l_i_weights)
    #   - Router_Init_Scale (tau)
    #   - Linear Router Gate (q_down_proj)
    #   - Selection Threshold (s_threshold_init)
    def __init__(self, config):
        super().__init__()

        # Yoink some things from config
        self.config = config
        self.scale = config.sqrt_hidden_size
        self.num_elastic_heads = self.config.num_elastic_heads

        # Summary Query Matrix size() : [hidden_size, num_router_latents]
        self.q_down_proj = nn.Linear(
            config.hidden_size,
            config.num_router_latents,
            bias = config.bias
        )

        # Latent Important Weights
        # size() [num_router_latents]
        self.l_i_weights = nn.Parameter(
            torch.ones(config.num_router_latents)
        )

        # Router_Init_Scale size() : [1]
        self.tau = nn.Parameter(torch.tensor(config.router_init_scale))

        # s_threshold size() : [1]
        self.s_threshold = nn.Parameter(torch.tensor(config.selection_threshold_init))

        # Linear Router Gate size() : [hidden_size, num_attention_heads - num_permanent_heads]
        self.q_up_proj = nn.Linear(
            config.hidden_size,
            config.num_elastic_heads,
            bias = config.bias
        )

    # Pass in only Hidden States 
    # Don't pass in attention mask bc theres no attention here (duh)
    def forward(self, hidden_states, current_step = None):

        # Set step to infinity if not passed in (during inference)
        step = current_step if current_step is not None else float('inf')

        # Write vars for cleaner code
        q_down_proj = self.q_down_proj
        l_i_weights = self.l_i_weights
        tau = self.tau
        q_up_proj = self.q_up_proj 
        scale = self.scale
        num_elastic_heads = self.num_elastic_heads
        
        # Norm Query Matrix
        # Requires .weight since the matrix was defined before
        q_down_proj = justnorm(q_down_proj.weight, dim = 1)

        # Multiply the hidden_state by Down projection (q_down_proj)
        # Call it "scanner"
        # Size: [b, s, hidden_size] * [hidden_size * num_router_latents] = [b, s, num_router_latents]
        scanner = F.linear(hidden_states, q_down_proj)

        # Apply Softmax to entire sequences (sequence level routing)
        scanner_softmax = F.softmax(scale * scanner, dim = 1)

        # Apply Transpose to allow for dimension matching
        # [b, s, num_router_latents] -> [b, num_router_latents, s]
        scanner_softmax = scanner_softmax.transpose(1,2)

        # Create Latent Vectors (Summary of the sequence in 4 vectors)
        # Size: [b,num_router_latents,s] * [b, s, hidden_size] = [b, num_router_latents, hidden_size]
        # Use bmm since the softmax is dynamic
        latents = torch.bmm(scanner_softmax, hidden_states)

        # Normalize the latents
        latents = justnorm(latents)

        # Scale latents by Learnable important parameters (l_i_weights)
        # Softmax them first
        l_i_weights = F.softmax(l_i_weights, dim = 0)

        # Apply l_i_weights to latents
        # Sum the Latents together
        # Size: [b, 1, hidden_size]
        pooled_latents = (latents * l_i_weights.view(1, -1, 1)).sum(dim=1)

        # Normalize again (per nGPT requirements)
        pooled_latents = justnorm(pooled_latents)
        
        # Normalize q_up_proj      
        # Requires .weight since the matrix was defined before
        q_up_proj = justnorm(q_up_proj.weight, dim = 1)

        # Multiply the latents by the classifer (q_up_proj)
        # Call it "class_scores"
        # Size: [b, 1, hidden_size] * [hidden_size, num_elastic_heads] = [b, 1, num_elastic_heads]
        class_scores = F.linear(pooled_latents, q_up_proj)

        # Multiply this by Tau (router_init_scale)
        class_scores = class_scores * tau * scale

        # Sigmoid Scores
        sigmoid_scores = torch.sigmoid(class_scores)

        # Selection Strategy:
        # Format: [b, num_elastic_heads, 1, 1]
        # Use Top-K for when we are in warm-up phase
        if (step < self.config.sparsity_warm_up_steps):

            # Call top_k on the router attention heads
            _, indices = sigmoid_scores.topk(num_elastic_heads, dim = -1)
            
            # Create One-hot encoding mask for top_k
            # size(): [num_elastic_heads]
            flat_mask = F.one_hot(indices, num_classes=sigmoid_scores.size(-1)).sum(dim=1).float()

        else:
            # Standard Threshold Phase: Hard 0.0 or 1.0 decisions
            flat_mask = (sigmoid_scores > self.s_threshold).float()

        # use_sigmoid_scaling = True: router_mask = Sigmoid values and 0s (Accuracy)
        # use_sigmoid_scaling = False: router_mask = 1s and 0s (Efficiency)
        if self.config.use_sigmoid_scaling:
            # Scale back to sigmoid scores
            flat_mask = flat_mask * sigmoid_scores

        # Apply STE for Dead Router Heads during backprop
        flat_mask = flat_mask.detach() - sigmoid_scores.detach() + sigmoid_scores

        # Add attention dimensions: [b, num_elastic_heads, 1, 1]
        router_mask = flat_mask.view(flat_mask.size(0), -1, 1, 1)

        # If permanent_heads are used, add the columns 
        # size(): [b, num_attention_heads, 1 , 1]
        if (self.config.num_permanent_heads > 0):
            permanent_head_scores = torch.ones(
                flat_mask.size(0), 
                self.config.num_permanent_heads, 
                1,
                1,
                device=router_mask.device, 
                dtype=router_mask.dtype
            )
            router_mask = torch.cat((permanent_head_scores, router_mask), dim = 1)

        # STE math going on ________________________________________________________________________________________ LOOK INTO THIS
        if self.training:
            P = sigmoid_scores.mean(dim=0).squeeze() 
            f = (sigmoid_scores > self.s_threshold).float().mean(dim=0).squeeze() # Best approx of hard_mask here
            self.aux_loss = self.num_elastic_heads * torch.sum(f * P)
        else:
            self.aux_loss = 0.0

        # Return Mask
        return router_mask



# RoPE Class
class RotaryEmbeddings(nn.Module):

    # Initialize the Following
    # rope_theta
    # max_position_embeddings
    # sin & cos table
    def __init__(self, dim, max_position_embeddings, rope_theta = 160000):
        super().__init__()

        # Define inverse of frequencies
        # size(): [dim/2]
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2).float() / dim))

        # Create position vector
        # size(): [max_position_embeddings]
        t = torch.arange(max_position_embeddings, dtype = inv_freq.dtype)

        freqs = torch.outer(t, inv_freq)

        freqs = torch.cat((freqs, freqs), dim = -1)


        # Save the Sine and Cosine
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())
    
    # Implement rotate_half (Allows for clean rotation mechanics)
    def rotate_half(self, x):

        # Take x as the first half
        x1 = x[..., : x.shape[-1] // 2]

        # Take y was the second half
        x2 = x[..., x.shape[-1] // 2 :]

        return torch.cat((-x2, x1), dim = -1)


    # Implement apply_rotary_embeddings 
    # Does RoPE
    # Expected input size: [b, num_attention_heads, seq_len, dim]
    # Output: [b, num_attention_heads, seq_len, dim]
    def forward(self, x):

        # Get token length
        seq_len = x.shape[-2]

        # Take a slice of the cos and sin tables
        x_cos = self.cos[:seq_len, ...].to(dtype=x.dtype)
        x_sin = self.sin[:seq_len, ...].to(dtype=x.dtype)

        # Return RoPE matrix
        return (x * x_cos) + (self.rotate_half(x) * x_sin)



# Self Attention
# Literally Just Self Attention
# QKV cross self attention
# Use RoPE
# Output Matrix
# Speicfics about training (masked training)
class HELMSelfAttention(nn.Module):

    # Initialize the following:
    #   - QKV matrix
    #   - Output matrix
    #   - Scaling vector sqk for q and k
    #   - RoPE Module
    def __init__(self, config):
        super().__init__()

        # Grabbing config values from convience
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.d_head = config.hidden_size // config.num_attention_heads
        self.ngpt_sqk_init_value = config.ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = config.ngpt_sqk_init_scale

        # QKV Matrix
        self.qkv = nn.Linear(
            config.hidden_size,
            config.hidden_size * 3,
            bias = config.bias 
        )

        # RoPE Module
        self.RoPE = RotaryEmbeddings(
            self.d_head, 
            config.max_position_embeddings, 
            config.rope_theta
        )

        # SQK scalers right after RoPE
        self.sqk = nn.Parameter(self.ngpt_sqk_init_scale*torch.ones(self.hidden_size, dtype=torch.float32))

        # Output Matrix
        self.output = nn.Linear(
            config.hidden_size,
            config.hidden_size,
            bias = config.bias
        )
    
    # Define Training
    def forward(self, hidden_states, attention_mask, router_mask):

        # Obtain projection from hidden_states onto QKV
        # size(): [b, seq_len, hidden_size * 3]
        qkv_proj = self.qkv(hidden_states)

        # Obtain Hidden Size
        batch_size, seq_len, hidden_size = hidden_states.size()

        # Split Projects
        # q, k, v size(): [b, seq_len, hidden_size]
        q, k, v = qkv_proj.split(hidden_size, dim=-1)

        # Define sqk for scaling q, k, and v
        # size(): [hidden_size]
        sqk = (self.sqk * (self.ngpt_sqk_init_value/self.ngpt_sqk_init_scale))
        # Resizing is required for when we element-wise multiply this by q and k matrice:s [1, num_attention_heads, 1, d_head] * [b, num_attention_heads, seq_len, hidden_size]
        # size(): [hidden_size]-> [1, num_attention_heads, 1, d_head]
        sqk = sqk.view(1, self.num_attention_heads, 1, self.d_head)

        # If Batch > 1 or model is in training mode: Cutting Losses and broadcast the mask
        if (batch_size > 1 or self.training):
            
            # Reshape q,k,v
            # q, k, v size(): [b, seq_len, num_attention_heads, d_head]
            q = q.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
            k = k.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
            v = v.view(batch_size, seq_len, self.num_attention_heads, self.d_head)

            # Reshape q,k,v
            # q, k, v size(): [b, num_attention_heads, seq_len, d_head]
            q = q.permute(0,2,1,3)
            k = k.permute(0,2,1,3)
            v = v.permute(0,2,1,3)

            # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor and justnorm to q and k (splice sqk too)
            q = sqk * q 
            k = sqk * k 

            # Apply Attention
            # Sclae by sqrt(dk)
            # A whole lot happens here. final size(): [b, num_attention_heads, seq_len, d_head]
            context_layer = F.scaled_dot_product_attention(
                q, k, v, 
                attn_mask=attention_mask,
                scale=math.sqrt(self.d_head)
            )

            # 🩹 THE ROBUST XLA BROADCAST FIX:
            if router_mask is not None:
                if router_mask.shape[1] != context_layer.shape[1]:
                    # We need to map 'V' views to 'H' heads (e.g., 18 views -> 20 heads)
                    # We treat the views like a 1D image and "stretch" it to the head count
                    # [B, V, 1, 1] -> [B, 1, V]
                    m = router_mask.squeeze(-1).squeeze(-1).unsqueeze(1)
                    # Interpolate to [B, 1, H]
                    m = F.interpolate(m, size=context_layer.shape[1], mode='nearest')
                    # Reshape back to [B, H, 1, 1]
                    router_mask = m.squeeze(1).unsqueeze(-1).unsqueeze(-1)
                
                # Explicit expansion for XLA safety
                # Apply Mask
                # size(): [b, num_attention_heads, seq_len, d_head]
                context_layer = context_layer * router_mask.expand_as(context_layer)

            # # Apply Mask
            # # size(): [b, num_attention_heads, seq_len, d_head]
            # context_layer = context_layer * router_mask

            # Reshape 
            # size(): [b, seq_len, num_attention_heads, d_head] 
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()
            
            # Flatten the last two dimensions: 
            # size(): [b, seq_len, num_attention_heads, d_head] 
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = self.output(context_reshaped)

        # Apply splicing logic and win on efficiency.
        else:

            # Find the heads that are on
            # nonzero(): [1, num_attention_heads, 1, 1] -> [num_active_heads, 1]
            # squeeze(): [num_active_heads, 1] -> [num_active_heads] (indices)
            active_indices = torch.nonzero(router_mask[0, :, 0, 0]).squeeze(-1)

            # Reshape q,k,v
            # q, k, v size(): [b, seq_len, num_attention_heads, hidden_size]
            q = q.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
            k = k.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
            v = v.view(batch_size, seq_len, self.num_attention_heads, self.d_head)

            # Reshape q,k,v
            # q, k, v size(): [b, num_attention_heads, seq_len, hidden_size]
            q = q.permute(0,2,1,3)
            k = k.permute(0,2,1,3)
            v = v.permute(0,2,1,3)

            # 2. Extract the parts used by the active heads
            # size(): [1, num_attention_heads, seq_len, d_head] ->  [1, num_active_heads, seq_len, d_head]
            q_sliced = q[:, active_indices, :, :]
            k_sliced = k[:, active_indices, :, :]
            v_sliced = v[:, active_indices, :, :]

            # Normalize q and k
            q_sliced = justnorm(q_sliced)
            k_sliced = justnorm(k_sliced)

            # Apply RoPE
            q_sliced = self.RoPE(q_sliced)
            k_sliced = self.RoPE(k_sliced)

            # Apply sqk scaling factor and justnorm to q and k 
            sqk_sliced = sqk[:, active_indices, :, :]
            q_sliced = sqk_sliced * q_sliced  
            k_sliced = sqk_sliced * k_sliced  

            # Flash Attention
            # size(): [b, num_active_heads, seq_len, d_head]
            context_sliced = F.scaled_dot_product_attention(
                q_sliced, k_sliced, v_sliced, 
                attn_mask=attention_mask,
                scale=math.sqrt(self.d_head)
            )

            # STE tie to the router
            # Note: If use_sigmoid_scaling = True: Scales the router mask back to the sigmoid values 
            # (since active indices were just indices of the values, not the real values)
            # If use_sigmooid_scaling = False, then multiplying by 1 does mathimatically nothing
            active_weights = router_mask[:, active_indices, :, :]
            context_sliced = context_sliced * active_weights

            # 5. Reshape for the output linear layer
            # [1, num_active, seq_len, d_head] -> [1, seq_len, num_active, d_head]
            context_reshaped = context_sliced.permute(0, 2, 1, 3).contiguous()
            
            # Flatten the last two dimensions: [1, seq_len, num_active * d_head]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # 6. Map the active head indices to their exact hidden dimension indices
            # Example: Head 1 with d_head=64 generates indices 64 through 127
            dim_offsets = torch.arange(self.d_head, device=hidden_states.device)
            active_dims = (active_indices.unsqueeze(1) * self.d_head + dim_offsets).view(-1)

            # 7. Slice the input columns of the output weight matrix
            # original shape [hidden_size, hidden_size] -> [hidden_size, num_active * d_head]
            sliced_weight = self.output.weight[:, active_dims]

            # 8. Perform the compressed functional linear projection
            context_layer = F.linear(context_reshaped, sliced_weight, bias=self.output.bias)
    
        # Return context_layer (normalization occurs in HELMMLP)
        return context_layer



# HELMMLP (FFN of nGPT architecture)
# All of this stays the same from the original nGPT paper
class HELMMLP(nn.Module):

    # Define the Following:
    #   - Constants from config (for convience?)
    #       * hidden_size
    #       * ngpt_alpha_value_attn
    #       * ngpt_alpha_scale_attn
    #       * ngpt_alpha_value_mlp
    #       * ngpt_alpha_scale_mlp
    #       * ngpt_suv_value
    #       * ngpt_suv_scale
    #   - Eigen learning rate after attention (attn_alpha)
    #   - Eigen learning rate after mlp (mlp_alpha)
    #   - MLP expansion layer (mlp_exp)
    #   - suv scaling vectors for SwiGLU (suv)
    #   - SiLU() activation (silu)
    #   - MLP projection layer (mlp_expand)
    def __init__(self, config):
        super().__init__()

        # Gather Config Values for convience
        self.hidden_size = config.hidden_size
        self.ngpt_alpha_value_attn = config.ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = config.ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = config.ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = config.ngpt_alpha_scale_mlp
        self.ngpt_suv_value = config.ngpt_suv_value
        self.ngpt_suv_scale = config.ngpt_suv_scale
        self.intermediate_size = config.intermediate_size

        # Alpha Eigen Update after Attention (1st Optimizer Step)
        self.attn_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_attn*torch.ones(self.hidden_size, dtype=torch.float32))

        # Alpha Eigen Update after MLP (2nd Optimizer Step)
        self.mlp_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_mlp*torch.ones(self.hidden_size, dtype=torch.float32))

        # MLP expansion layer
        self.mlp_exp = nn.Linear(
            self.hidden_size, 
            2 * self.intermediate_size, 
            bias = config.bias
        )

        # suv scaling vectors during SwiGLU 
        self.suv = torch.nn.Parameter(self.ngpt_suv_scale*torch.ones(2 * self.intermediate_size, dtype=torch.float32))

        # Define SiLU()
        self.silu = nn.SiLU()

        # MLP projection layer (shrink)
        self.mlp_proj  = nn.Linear(
            self.intermediate_size,
            self.hidden_size,
            bias=config.bias
        )
    
    # Peform MLP from the output of the output matrix to the end of the transformer block
    def forward(self, hidden_states, hidden_states_attention):

        # Even more convience
        hidden_size = self.hidden_size
        ngpt_alpha_value_attn = self.ngpt_alpha_value_attn
        ngpt_alpha_scale_attn = self.ngpt_alpha_scale_attn
        ngpt_alpha_value_mlp = self.ngpt_alpha_value_mlp
        ngpt_alpha_scale_mlp = self.ngpt_alpha_scale_mlp
        ngpt_suv_value = self.ngpt_suv_value
        ngpt_suv_scale = self.ngpt_suv_scale

        # Mostly Lifted from the nGPT model.py

        # Define the eigen learning rate 
        # alpha >=0
        # size(): [hidden_size]
        lr = self.attn_alpha * (ngpt_alpha_value_attn / ngpt_alpha_scale_attn)
        lr = torch.abs(lr)
        
        # Apply Normalization to hidden states before and after attention
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states)
        B_norm = justnorm(hidden_states_attention)
            
        # h = Norm(h + alpha_a * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt1 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt1 = justnorm(hidden_states_opt1)

        # Get u and v matrices by multiplying by mlp_exp
        # size(): [b, seq_len, hidden_size] * [hidden_size, 2 * intermediate_size] = [b, seq_len, 2 * intermediate_size]
        uv = self.mlp_exp(hidden_states_opt1)

        # prepare scaling vector suv
        # size(): [intermediate_size * 2] (remember, they are concatenated)
        suv = self.suv * (ngpt_suv_value/ngpt_suv_scale) * (hidden_size ** 0.5)
        
        # element-wise uv by scaling vector suv
        # size(): [b, seq_len, 2 * intermediate_size]
        uv = suv * uv  

        # Chunk uv into u and v
        # both size(): [b, seq_len, intermediate_size]
        u, v = torch.chunk(uv, 2, dim=-1)

        # Apply u * silu(v), the whole point of SwiGLU (element-wise)
        # size(): [b, seq_len, intermediate_size]
        x_mlp = u * self.silu(v)

        # Project x_mlp to the mlp_proj layer (shrink)
        # size(): [b, seq_len, intermediate_size] * [intermediate_size, hidden_size] = [b, seq_len, hidden_size]
        h_mlp = self.mlp_proj(x_mlp)

        # Define the eigen learning rate 
        # alpha >=0
        # size(): [hidden_size]
        lr = self.mlp_alpha * (ngpt_alpha_value_mlp / ngpt_alpha_scale_mlp) 
        lr = torch.abs(lr)

        # Apply Normalization to hidden states after attention and after mlp
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states_opt1)
        B_norm = justnorm(h_mlp)

        # h = Norm(h + alpha_m * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt2 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt2 = justnorm(hidden_states_opt2)

        # Return new hidden_state
        return hidden_states_opt2



# HELMBLOCK = HELMMultiViewRouter + HELMSelfAttention (which defines RotaryEmbeddigs) + HELMMLP
# This is 1 transformer layer
class HELMBlock(nn.Module):

    # Define the Following:
    #   - HELMMultiViewRouter
    #   - HELMSelfAttention
    #   - HELMMLP
    def __init__(self, config):
        super().__init__()
        self.mlt_vw_rtr = HELMMultiViewRouter(config)
        self.attn = HELMSelfAttention(config)
        self.mlp = HELMMLP(config)
    
    # Define the forward pass
    # Extra: Return the aux_loss from the router
    def forward(self, hidden_states, attention_mask, current_step = None):
        router_mask = self.mlt_vw_rtr(hidden_states, current_step)
        aux_loss = self.mlt_vw_rtr.aux_loss
        attn_output = self.attn(hidden_states, attention_mask, router_mask)
        layer_output = self.mlp(hidden_states, attn_output)
        return layer_output, aux_loss



# HELMModel - HELM without the head 
class HELMModel(nn.Module):

    # Define the following:
    #   - HELMEmbedding
    #   - HELMBlock
    def __init__(self, config):
        super().__init__()

        # Embedding layer
        self.embedding = HELMEmbedding(config)

        # Transformer blocks
        self.blocks = nn.ModuleList(
            [HELMBlock(config) for _ in range(config.num_hidden_layers)]
        )
    
    # Forward Pass
    def forward(self, input_ids, attention_mask, current_step = None):


        # Reshape Attention Mask to be 4D for SDPA [batch_size, 1, 1, seq_len]
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2).float()

        # Then make attend = 0 and mask = -inf to allow for Flash Attention compatitbility
        attention_mask = attention_mask.masked_fill(attention_mask == 0, float('-inf'))
        attention_mask = attention_mask.masked_fill(attention_mask == 1, 0.0)

        # Pass input_ids through the input
        embeddings = self.embedding(input_ids)

        # Set Embeddings to be hidden_states
        hidden_states = embeddings 

        # Accumulate total_aux_loss

        total_aux_loss = 0

        # Run Tranformer Blocks
        # Remember to pass in current_step #
        for block in self.blocks:
            hidden_states, aux_loss = block(hidden_states, attention_mask, current_step)
            total_aux_loss += aux_loss

        # Return hidden state (feature extraction / context location prediction)
        return hidden_states, total_aux_loss



# HELMModelforMaskedLM
class HELMForMaskedLM(PreTrainedModel):

    # Define the Config for the HF push_to_hub() function
    config_class = HELMConfig

    # Define the Following:
    #   - HELMModel
    #   - classifier
    #   - Head layer scaling vector
    def __init__(self, config):
        super().__init__(config)

        # Define from Config
        self.ngpt_sz_init_value = config.ngpt_sz_init_value
        self.ngpt_sz_init_scale = config.ngpt_sz_init_scale

        # Define the Model
        self.model = HELMModel(config)

        # Define the head Layer
        self.classifier = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias = config.bias
        )

        # Define the head layer scaling vetor 
        self.sz = nn.Parameter(torch.ones(config.vocab_size, dtype=torch.float32))

        # HF Function to call _init_weights() function
        self.post_init()

    # Initialize weights (pulled from ngpt model.py)
    def _init_weights(self, module):

        # If it's an nn.Linear, initialize it with the initializer_range
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        # If it's an nn.Linear, initialize it with the initializer_range also)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)

    
    # Forward pass
    def forward(self, input_ids, attention_mask, current_step = None):

        # Gather Context from the model
        features, total_aux_loss = self.model(input_ids, attention_mask, current_step)

        # Scale / prepare sz
        sz = self.sz * (self.ngpt_sz_init_value / self.ngpt_sz_init_scale)

        # project features onto classifer
        unscaled_logits = self.classifier(features)

        # Scale the logits with sz
        logits = sz * unscaled_logits

        # Return Logits
        return logits, total_aux_loss

    
